Once libraries were properly tested, The html from the CCNY page is requested through requests, and using BeautifulSoup, the html data is pulled and ready to use for us.

In [14]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime

#Requesting from CCNY Calendar 
CityWeb = requests.get("https://www.ccny.cuny.edu/registrar/fall")
CitySoup = BeautifulSoup(CityWeb.text,'html.parser')
print(CitySoup) #shows the html of the CCNY webpage, the latest output printed a placeholder string for the sake of order.

This Looks Cleaner :D


After observing the data from the html file is noticed that the calendar we are trying to scrape from is under table so variable CityTable will encapsulate whatever is enclosed by table.
Looking further it was found that th holds the header of the table, while td holds the content below these headers, the prints show what is actually extracted from this filtering.

In [65]:
CityTable = CitySoup.find("table")

Headers = [th.get_text(strip=True) for th in table.find_all("th")] #storing in Headers the three main headers from th
print(Headers)

Content = [td.get_text(strip=True) for td in table.find_all("td")] #storing in content the whole list of things from td, fixed from previous commit that used p, leading to bugs
print('\n') 
print(Content)


['DATES', 'DAYS', '']


['August 01', 'Sunday', 'Application for degree for January and February 2022 begins', 'August 18', 'Wednesday', 'Last day to apply for Study Abroad', 'August 24', 'Tuesday', 'Last day of Registration;Last day to file ePermit for the Fall 2021;Last day to drop classes for 100% tuition refund;', 'August 25', 'Wednesday', 'Start of Fall Term;Classes begin;Initial Registration Appeals begin;', 'August 25 - 31', 'Wednesday - Tuesday', 'Change of program period; late fees apply', 'August 26', 'Thursday', 'Last day for Independent Study', 'August 28', 'Saturday', 'First day of Saturday Classes', 'August 31', 'Tuesday', "Last day to add a class to an existing enrollment;Last day for 75% tuition refund;Financial Aid Certification Enrollment Status date;Last day to apply for Audit option;Last day to drop without the grade of 'WD';Initial Registration Appeals end;", 'September 01', 'Wednesday', "Verification of Enrollment rosters available to faculty;Course Withdrawal dro

After Getting the outputs from above we can notice two things, Our column list has the text column with no name and the list of content is not divided to match the columns, then i organized
content into ContentOrganized to make it match with Header.

In [101]:
Headers[1]='WEEKDAY'
Headers[2]='TEXT'

ContentOrganized = [Content[i:i+3] for i in range(0, len(Content), 3)] #for loop that organizes Content into ContentOrganized
CityDFrame = pd.DataFrame(ContentOrganized, columns=Headers)
print(CityDFrame)

                DATES               WEEKDAY  \
0           August 01                Sunday   
1           August 18             Wednesday   
2           August 24               Tuesday   
3           August 25             Wednesday   
4      August 25 - 31   Wednesday - Tuesday   
5           August 26              Thursday   
6           August 28              Saturday   
7           August 31               Tuesday   
8        September 01             Wednesday   
9   September 03 - 08    Friday - Wednesday   
10       September 06                Monday   
11       September 09              Thursday   
12       September 14               Tuesday   
13       September 15             Wednesday   
14  September 15 - 16  Wednesday - Thursday   
15       September 23              Thursday   
16       September 24                Friday   
17         October 01                Friday   
18         October 08                Friday   
19         October 11                Monday   
20        Nov

The DataFrame now work properly, but we still have a numbered index instead of Date being the index, so with .set_index() function from pandas we set DATES as our Index, Ending with a proper table of CCNY's 2021 calendar.
To fulfill th Lab's requirement, function DTimeConversion was created to organize, remove and include whatever is needed to transform every date in a python datetime.

In [102]:
def DTimeConversion(fecha):
    if ',' in fecha:  #in case it's like the last date which already has the year included
        return pd.to_datetime(fecha)
    
    if '-' in fecha: #in case is an interval just take the initial date
        fecha = fecha.split('-')[0].strip()
        
    return pd.to_datetime(fecha + ' 2021') #add 2021 then convert into datetime.

CityDFrame["DATES"] = CityDFrame["DATES"].apply(DTimeConversion)
CityDFrame.set_index("DATES", inplace=True)
print(CityDFrame)

                         WEEKDAY  \
DATES                              
2021-08-01                Sunday   
2021-08-18             Wednesday   
2021-08-24               Tuesday   
2021-08-25             Wednesday   
2021-08-25   Wednesday - Tuesday   
2021-08-26              Thursday   
2021-08-28              Saturday   
2021-08-31               Tuesday   
2021-09-01             Wednesday   
2021-09-03    Friday - Wednesday   
2021-09-06                Monday   
2021-09-09              Thursday   
2021-09-14               Tuesday   
2021-09-15             Wednesday   
2021-09-15  Wednesday - Thursday   
2021-09-23              Thursday   
2021-09-24                Friday   
2021-10-01                Friday   
2021-10-08                Friday   
2021-10-11                Monday   
2021-11-01                Monday   
2021-11-02               Tuesday   
2021-11-04              Thursday   
2021-11-06              Saturday   
2021-11-23               Tuesday   
2021-11-25     Thursday - Su